 **Import Libraries**

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

**Mount Google Drive**

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Load the **Dataset**

In [5]:
# Load the Bike Sharing hourly dataset
df = pd.read_csv("/content/drive/MyDrive/Group 11 /hour.csv")
df

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17374,17375,2012-12-31,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642,11,108,119
17375,17376,2012-12-31,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642,8,81,89
17376,17377,2012-12-31,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642,7,83,90
17377,17378,2012-12-31,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343,13,48,61


**Basic Data Understanding**

In [12]:
# Check the basic structure of the dataset
# Check the shape of the dataset
df.shape

(17379, 17)

In [13]:
# Check the column names
df.columns

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='object')

In [14]:
# Check missing values
df.isnull().sum()

,0
instant,0
dteday,0
season,0
yr,0
mnth,0
hr,0
holiday,0
weekday,0
workingday,0
weathersit,0


In [15]:
# Check duplicate rows
df.duplicated().sum()

np.int64(0)

In [16]:
# Check data types
df.dtypes

,0
instant,int64
dteday,object
season,int64
yr,int64
mnth,int64
hr,int64
holiday,int64
weekday,int64
workingday,int64
weathersit,int64


**Convert Date Column**

In [17]:
# Convert the date column into datetime format
df["dteday"] = pd.to_datetime(df["dteday"])
print(df["dteday"].dtype)

datetime64[ns]


**Data Validation**

In [18]:
# Check important columns for invalid values

checks = {
    "Invalid hr": ((df["hr"] < 0) | (df["hr"] > 23)).sum(),
    "Invalid season": ((df["season"] < 1) | (df["season"] > 4)).sum(),
    "Invalid month": ((df["mnth"] < 1) | (df["mnth"] > 12)).sum(),
    "Invalid temperature": ((df["temp"] < 0) | (df["temp"] > 1)).sum(),
    "Invalid humidity": ((df["hum"] < 0) | (df["hum"] > 1)).sum(),
    "Invalid windspeed": ((df["windspeed"] < 0) | (df["windspeed"] > 1)).sum()
}

print(pd.Series(checks))

Invalid hr             0
Invalid season         0
Invalid month          0
Invalid temperature    0
Invalid humidity       0
Invalid windspeed      0
dtype: int64


**Detect Potential Outliers**

In [19]:
# Detect potential outliers in total bike rentals (cnt)
# Outliers are detected but not automatically deleted

Q1 = df["cnt"].quantile(0.25)
Q3 = df["cnt"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[
    (df["cnt"] < lower_limit) |
    (df["cnt"] > upper_limit)
]

print("Number of potential outliers:", len(outliers))

Number of potential outliers: 505


**Sort Data Chronologically**

In [20]:
# Sort the dataset by date and hour
df = df.sort_values(
    by=["dteday", "hr"]
).reset_index(drop=True)

print(df.head())

   instant     dteday  season  yr  mnth  hr  holiday  weekday  workingday  \
0        1 2011-01-01       1   0     1   0        0        6           0   
1        2 2011-01-01       1   0     1   1        0        6           0   
2        3 2011-01-01       1   0     1   2        0        6           0   
3        4 2011-01-01       1   0     1   3        0        6           0   
4        5 2011-01-01       1   0     1   4        0        6           0   

   weathersit  temp   atemp   hum  windspeed  casual  registered  cnt  
0           1  0.24  0.2879  0.81        0.0       3          13   16  
1           1  0.22  0.2727  0.80        0.0       8          32   40  
2           1  0.22  0.2727  0.80        0.0       5          27   32  
3           1  0.24  0.2879  0.75        0.0       3          10   13  
4           1  0.24  0.2879  0.75        0.0       0           1    1  


In [22]:
# Save the cleaned dataset to Google Drive
df.to_csv("/content/drive/MyDrive/Group 11 /cleaned_bike_sharing.csv", index=False)

**Chronological Train-Test Split**

In [23]:
# Split the data chronologically into 80% training and 20% testing
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Training rows: 13903
Testing rows: 3476


**Prepare Features, Target & Prevent Data Leakage**

In [24]:
# Prepare training and testing features and target

# Remove target, leakage columns, ID and date columns
# casual and registered are removed because they contribute to cnt

X_train = train_df.drop(
    columns=["cnt", "casual", "registered", "instant", "dteday"]
)

y_train = train_df["cnt"]

X_test = test_df.drop(
    columns=["cnt", "casual", "registered", "instant", "dteday"]
)

y_test = test_df["cnt"]

**Define Numerical and Categorical Features**

In [25]:
# Define numerical features

numeric_features = [
    "temp",
    "atemp",
    "hum",
    "windspeed"
]

# Define categorical features

categorical_features = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit"
]

**Preprocessing: Imputation, Scaling & One-Hot Encoding**

In [26]:
# Import preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing with One-Hot Encoding
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

**Apply Preprocessing**

In [27]:
# Fit preprocessing only on training data
# Then apply the same preprocessing to test data

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape:", X_test_processed.shape)

Training processed shape: (13903, 61)
Testing processed shape: (3476, 61)


**Create Processed DataFrames**

In [28]:
# Get the names of the processed features
feature_names = preprocessor.get_feature_names_out()

# Convert processed training data into a DataFrame
X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray(),
    columns=feature_names
)

# Convert processed testing data into a DataFrame
X_test_processed_df = pd.DataFrame(
    X_test_processed.toarray(),
    columns=feature_names
)

# Add the target variable
X_train_processed_df["cnt"] = y_train.values
X_test_processed_df["cnt"] = y_test.values

**Save Training and Testing Feature Matrices**

In [30]:
# Save processed training and testing datasets

X_train_processed_df.to_csv(
    "/content/drive/MyDrive/Group 11 /train_features.csv",
    index=False
)

X_test_processed_df.to_csv(
    "/content/drive/MyDrive/Group 11 /test_features.csv",
    index=False
)

**Create Data Quality & Audit Log**

In [31]:
# Create a data quality and audit log

quality_log = {
    "Rows": len(df),
    "Columns": len(df.columns),
    "Missing Values": int(df.isnull().sum().sum()),
    "Duplicate Rows": int(df.duplicated().sum()),
    "Potential Outliers": len(outliers),
    "Invalid Hour Values": int(
        ((df["hr"] < 0) | (df["hr"] > 23)).sum()
    )
}

quality_df = pd.DataFrame([quality_log])

print(quality_df)

    Rows  Columns  Missing Values  Duplicate Rows  Potential Outliers  \
0  17379       17               0               0                 505   

   Invalid Hour Values  
0                    0  


**Save Data Quality Log**

In [32]:
# Save the data quality log to Google Drive

quality_df.to_csv(
    "/content/drive/MyDrive/Group 11 /data_quality_log.csv",
    index=False
)